# Proyecto NFL: entrenamiento y evaluación

<a href="https://colab.research.google.com/github/<usuario>/NFL/blob/main/fuentes/entrena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

¿Las pruebas del NFL Combine pronostican cómo se mueve y rinde un prospecto en los partidos de temporada regular,
y eso dice algo sobre quién llega a ser titular? Este notebook ejecuta el proyecto completo, de los datos
originales a los modelos registrados en MLflow:

1. prepara el entorno (clona el repositorio en Colab e instala las dependencias fijadas);
2. revisa los datos originales contra el manifiesto y el contrato de datos;
3. limpia los datos y construye las métricas de movimiento y de rendimiento en el campo (`datos_prep.py`);
4. entrena y evalúa los modelos con validación cruzada anidada y los registra en MLflow (`train.py`);
5. muestra las métricas, las pruebas estadísticas y las figuras;
6. carga los modelos registrados y los usa para predecir.

El notebook llama exactamente al mismo código que `make todo`, así que da los mismos números. El reporte con la
interpretación está en `CHANGELOG.md` y la guía de reproducción en `README.md`.

| Modo | Qué hace | Duración en Colab |
| --- | --- | --- |
| `rapida` | Todo el flujo con validación cruzada y rejillas mínimas; escribe en carpetas propias (`datos/rapida`, `reportes/rapida`, `mlflow/rapida.db`) y no toca los resultados de entrega | unos 10 min |
| `completa` | La corrida de entrega (5 × 3 pliegues externos, rejillas completas); reescribe `datos/datos_limp`, `reportes/` y agrega versiones al registro `mlflow/mlflow.db` | 60–90 min |
| `solo_cargar` | No entrena: muestra las métricas y figuras de la entrega y carga los 17 modelos ya registrados | 1–2 min |

## 1. Configuración

Escriba la URL de su repositorio de GitHub y elija el modo. Si el repositorio es privado, Colab necesita
credenciales para clonarlo: use una URL con un token de acceso personal
(`https://<token>@github.com/<usuario>/NFL.git`) o haga público el repositorio. Si abre el notebook desde una
copia local del repositorio (Jupyter o VS Code), la URL no se usa: el notebook encuentra la carpeta del
proyecto solo.

In [ ]:
REPO_URL = "https://github.com/<usuario>/NFL.git"   # cambie <usuario> por su cuenta de GitHub
MODO = "rapida"                                     # "rapida", "completa" o "solo_cargar"

assert MODO in ("rapida", "completa", "solo_cargar"), "MODO debe ser 'rapida', 'completa' o 'solo_cargar'"

## 2. Preparar el entorno

En Colab se clona el repositorio en `/content/NFL` y se instalan las versiones fijadas en `requirements.txt`.
Si la instalación cambia bibliotecas que el entorno ya tenía cargadas en memoria, el entorno se reinicia una
vez de forma automática: en ese caso vuelva a ejecutar todas las celdas (*Entorno de ejecución → Ejecutar
todas*); la segunda vez la instalación se salta.

Si la versión de Python de Colab no tiene alguna de las versiones fijadas, se instalan las versiones más
recientes compatibles. El flujo funciona igual, aunque los últimos decimales de algunas métricas podrían variar.

In [ ]:
import os, sys, subprocess, pathlib, logging

EN_COLAB = "google.colab" in sys.modules


def buscar_raiz(inicio: pathlib.Path):
    """Carpeta del proyecto: la primera, hacia arriba, que contiene fuentes/datos_prep.py."""
    for c in [inicio, *inicio.parents]:
        if (c / "fuentes" / "datos_prep.py").exists():
            return c
    return None


RAIZ = buscar_raiz(pathlib.Path.cwd())
if RAIZ is None:
    destino = pathlib.Path("/content/NFL") if EN_COLAB else pathlib.Path.cwd() / "NFL"
    if not destino.exists():
        assert "<usuario>" not in REPO_URL, "Escriba la URL de su repositorio en REPO_URL (celda 1)"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
    RAIZ = destino
os.chdir(RAIZ)
print("Carpeta del proyecto:", RAIZ)

marca = RAIZ / ".dependencias_instaladas"
if EN_COLAB and not marca.exists():
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    if r.returncode != 0:
        print("Las versiones fijadas no están disponibles para esta versión de Python; "
              "se instalan las más recientes compatibles.")
        paquetes = [l.split("==")[0] for l in open("requirements.txt", encoding="utf-8")
                    if l.strip() and not l.startswith("#")]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
    marca.touch()
    cargadas = [m for m in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "xgboost") if m in sys.modules]
    if cargadas:
        print("Se reinicia el entorno para usar las versiones instaladas de", cargadas,
              "-> vuelva a ejecutar todas las celdas.")
        os.kill(os.getpid(), 9)
print("Entorno listo.")

In [ ]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
sys.path.insert(0, str(RAIZ / "fuentes"))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

import warnings
import pandas as pd
from IPython.display import Image, Markdown, display

import datos_prep
import train

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

CONFIG = "configuracion/rapida.yaml" if MODO == "rapida" else "configuracion/config.yaml"
cfg = datos_prep.load_config(CONFIG)
cfg_entrega = datos_prep.load_config("configuracion/config.yaml")
datos_prep.set_global_seed(cfg.seed)

# MLflow guarda rutas absolutas: se apuntan a esta carpeta (no hace nada si ya apuntan aquí).
print("Celdas del almacén MLflow reubicadas:", train.relocate_store(cfg_entrega))
print(f"Versión del flujo {datos_prep.__version__} | modo {MODO} | configuración {CONFIG} | semilla {cfg.seed}")

## 3. Datos originales y contrato de datos

Los diez archivos originales están en `datos/datos_ini/`, comprimidos con gzip para respetar el límite de
100 MB por archivo de GitHub. El manifiesto guarda el hash SHA-256 del contenido de cada uno; si algún archivo
cambió, la celda lo indica y la versión de datos (`data_version`) ya no es la del reporte.

El contrato de datos (`configuracion/contrato_datos.yaml`) es la especificación del proveedor en forma
revisable: tipos, rangos, claves, claves foráneas y cobertura. Los errores detienen el flujo; las desviaciones
conocidas son advertencias que manejan las reglas de negocio.

In [ ]:
import json

manifiesto = json.loads(cfg_entrega.path("manifest").read_text(encoding="utf-8"))
distintos = datos_prep.verify_manifest(cfg_entrega)
print("data_version:", manifiesto["data_version"],
      "| todos los archivos coinciden con el manifiesto" if not distintos else f"| NO coinciden: {distintos}")

archivos = pd.DataFrame([
    {"archivo": nombre, "guardado como": e.get("stored_as", nombre), "registros": e["rows"],
     "MB sin comprimir": round(e["bytes"] / 1e6, 1), "MB comprimido": round(e.get("stored_bytes", e["bytes"]) / 1e6, 1)}
    for nombre, e in manifiesto["files"].items()
])
display(archivos)

In [ ]:
resumen = datos_prep.stage_contract(cfg)
print(f"Contrato: {resumen['status']} — {resumen['n_checks']} revisiones, "
      f"{resumen['n_errors']} errores, {resumen['n_warnings']} advertencias")

reporte = pd.read_csv(cfg.path("metrics") / "data_contract_report.csv")
advertencias = reporte[(reporte.severity == "warning") & (reporte.result == "fail")]
display(advertencias[["table", "column", "check", "detail", "handling"]].head(32))

## 4. Limpieza y preparación de los datos (`datos_prep.py`)

La etapa de datos aplica las reglas de negocio R1–R16 (rangos válidos de las pruebas, turno 260 para los no
seleccionados, anclaje al snap, topes físicos del tracking, mínimo de jugadas por jugador, temporada regular,
jugadas anuladas, downs 1–4, fiabilidad del tracking de pruebas) y construye:

- la tabla de prospectos con el objetivo `is_starter` (inició el 50% o más de los partidos disponibles);
- las métricas por jugada del tracking de partido y el Snap Burst Index por jugador;
- las métricas de campo sobre lo esperado por jugador (`player_play`);
- las variables fiables del tracking de las pruebas del Combine.

En modo `solo_cargar` se leen los datos limpios que ya vienen en `datos/datos_limp/`.

In [ ]:
if MODO == "solo_cargar":
    d = datos_prep.load_processed(cfg)
else:
    d = datos_prep.stage_data(cfg)   # unos 2 minutos

auditoria = json.loads((cfg.path("metrics") / "data_audit.json").read_text(encoding="utf-8"))
claves = ["frames_in", "frames_out", "plays_kept", "players_tracked", "players_profiled", "ct_attempts_kept",
          "pp_rows_regular_season", "players_with_perf_index", "play_metric_anomalies", "contract_status"]
display(pd.Series({k: auditoria.get(k) for k in claves if k in auditoria}, name="valor").to_frame())

p = d["prospects"]
print(f"{len(p)} prospectos, {int(p.is_starter.sum())} titulares ({p.is_starter.mean():.0%})")
display(p.groupby("combine_position", observed=True)
         .agg(prospectos=("nfl_id", "size"), titulares=("is_starter", "sum"),
              cuarenta_medio=("forty", "mean"), pruebas_corridas=("n_drills_run", "mean"))
         .round(2))

In [ ]:
perfiles = d["profiles"]
print(f"{len(perfiles)} jugadores con 3+ jugadas registradas; mediana de jugadas por jugador:",
      int(perfiles.n_plays.median()))
display(Markdown("**Proporción de prospectos sin cada prueba, por posición** (las faltantes se imputan dentro "
                 "de cada pliegue con la mediana de la posición, y un indicador marca la falta):"))
display(pd.read_csv(cfg.path("metrics") / "drill_missingness_by_position.csv", index_col=0).round(2))

## 5. Entrenamiento y evaluación (`train.py`)

La etapa `modelos` ejecuta, en este orden:

1. **Traducción**: correlaciones prueba → partido y prueba → campo dentro de la posición (con corrección FDR),
   fiabilidad de cada métrica (ICC), mediación en la línea defensiva y los GLM de titulares.
2. **Regresión**: Combine → SBI (todas las posiciones, línea defensiva, receptores) y Combine → rendimiento de
   campo (cuatro tareas). Compite la media por posición (referencia) contra Ridge, Random Forest y XGBoost.
3. **Clasificación**: ¿quién llega a titular? Regresión logística, Random Forest y XGBoost sobre conjuntos de
   variables anidados (Combine; + draft; + movimiento; + campo; + tracking de pruebas).
4. **Pruebas estadísticas** sobre los mismos pliegues: t corregida para remuestreo (Nadeau-Bengio), Wilcoxon
   con Holm, Friedman y DeLong.
5. **Registro en MLflow**: cada corrida guarda parámetros, etiquetas (`pipeline_version`, `data_version`,
   `git_commit`, semilla), puntajes por pliegue y predicciones fuera de pliegue; los campeones y retadores se
   reentrenan con todas las filas y se registran con alias.

La validación cruzada es anidada: 5 pliegues × 3 repeticiones por fuera y 3 pliegues por dentro para elegir
hiperparámetros (en modo `rapida`: 2 × 1 y 2). La imputación y el escalado se aprenden dentro de cada pliegue.

In [ ]:
if MODO != "solo_cargar":
    train.main(["--config", CONFIG, "--etapa", "modelos"])   # rápida: unos 6 min; completa: 55–85 min
    train.main(["--config", CONFIG, "--etapa", "figuras"])
else:
    print("Modo solo_cargar: se usan las métricas y figuras de la corrida de entrega.")

## 6. Resultados

### 6.1 Regresión: ¿pronostica el Combine el movimiento y el rendimiento en partido?

Para cada población: la referencia (media por posición) y el mejor modelo según el RMSE medio de los pliegues
externos, con el R² agrupado fuera de pliegue y su intervalo bootstrap de 95%.

In [ ]:
M, F = cfg.path("metrics"), cfg.path("figures")

reg = pd.read_csv(M / "regression_summary.csv")
reg = reg[~reg.population.str.endswith("_ct")]
filas = []
for pob, g in reg.groupby("population", sort=False):
    base = g[g.model == "position_mean"].iloc[0]
    mejor = g[g.model != "position_mean"].sort_values("rmse_mean").iloc[0]
    filas.append({"población": pob, "objetivo": base.target, "n": int(base.n), "mejor modelo": mejor.model,
                  "RMSE referencia": base.rmse_mean, "RMSE modelo": mejor.rmse_mean,
                  "cambio %": 100 * (mejor.rmse_mean / base.rmse_mean - 1),
                  "R² agrupado": mejor.oof_r2, "IC 95%": f"[{mejor.oof_r2_ci_low:.2f}, {mejor.oof_r2_ci_high:.2f}]"})
tabla_reg = pd.DataFrame(filas)
display(tabla_reg.round(3))

pruebas = pd.read_csv(M / "regression_pairwise_tests.csv")
vs_ref = pruebas[(pruebas.metric == "rmse") & (pruebas.model_a == "position_mean")
                 & ~pruebas.population.str.endswith("_ct")]
display(Markdown("**Cada modelo contra la referencia** (diferencia de RMSE > 0 = el modelo tiene menos error; "
                 "p con corrección de Holm):"))
display(vs_ref[["population", "model_b", "mean_diff_a_minus_b", "t_corrected", "p_corrected_t_holm",
                "p_wilcoxon_holm"]].round(4))

In [ ]:
for f, titulo in [("fig10_regression_cv.png", "Error de cada modelo del SBI por pliegue"),
                  ("fig18_onfield_regression.png", "Combine → rendimiento en el campo")]:
    if (F / f).exists():
        display(Markdown(f"**{titulo}**"), Image(filename=str(F / f), width=900))

### 6.2 Clasificación: ¿quién llega a titular?

ROC-AUC medio de los pliegues externos por población, conjunto de variables y modelo. El capital de draft es
lo que más mueve el AUC; el movimiento en partido y el tracking de pruebas casi no lo cambian.

In [ ]:
clf = pd.read_csv(M / "classification_summary.csv")
display(clf.pivot_table(index=["population", "feature_set"], columns="model", values="roc_auc_mean",
                        sort=False).round(3))

display(Markdown("**DeLong** (dos AUC sobre las mismas predicciones fuera de pliegue):"))
display(pd.read_csv(M / "classification_delong.csv").round(4))

campeon = clf[(clf.population == "all_prospects") & (clf.feature_set == "combine_draft")]
display(Markdown("**Modelo de titulares antes del draft (Combine + draft)**, métricas con umbral 0.5:"))
display(campeon[["model", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "f1_mean", "recall_mean",
                 "precision_mean", "brier_mean"]].round(3))

In [ ]:
for f, titulo in [("fig11_classification_auc.png", "AUC por conjunto de variables"),
                  ("fig12_roc_curves.png", "Curvas ROC fuera de pliegue"),
                  ("fig13_feature_importance.png", "Importancia de variables")]:
    if (F / f).exists():
        display(Markdown(f"**{titulo}**"), Image(filename=str(F / f), width=900))

### 6.3 Tracking de las pruebas del Combine: ¿suma algo?

El mejor modelo con y sin las cinco variables fiables del tracking de pruebas, en los mismos jugadores y
pliegues.

In [ ]:
ct = pd.read_csv(M / "ct_best_vs_best.csv")
cols = ["task", "n", "model_without", "model_with", "r2_without", "r2_with", "r2_diff", "r2_p_corrected_t",
        "roc_auc_without", "roc_auc_with", "roc_auc_p_corrected_t", "p_delong"]
display(ct[[c for c in cols if c in ct.columns]].round(4))
if (F / "fig20_ct_value.png").exists():
    display(Image(filename=str(F / "fig20_ct_value.png"), width=900))

### 6.4 Brecha de traducción en la línea defensiva

SBI observado en partido menos el SBI que predice el perfil del Combine (fuera de pliegue). Positivo = el
jugador arranca más rápido de lo que midió; negativo = su perfil dice que puede arrancar más rápido de lo que lo
hace, lo que apunta a la técnica.

In [ ]:
brecha = pd.read_csv(M / "translation_gap.csv")
dl = brecha[(brecha.population == "defensive_line") & (brecha.n_plays >= 10)].sort_values("translation_gap")
cols = ["display_name", "nfl_position", "draft_year", "draft_round", "n_plays", "sbi", "predicted_sbi", "translation_gap"]
display(pd.concat([dl.tail(5)[::-1], dl.head(5)])[cols].round(1))
if (F / "fig14_translation_gap.png").exists():
    display(Image(filename=str(F / "fig14_translation_gap.png"), width=900))

### 6.5 Todas las figuras

Las 21 figuras del reporte quedan en la carpeta de figuras de la configuración (`reportes/figuras/` en la
corrida de entrega).

In [ ]:
figuras = sorted(F.glob("fig*.png"))
print(len(figuras), "figuras en", F)
MOSTRAR_TODAS = False   # cambie a True para verlas todas aquí
for f in figuras if MOSTRAR_TODAS else []:
    display(Markdown(f"**{f.name}**"), Image(filename=str(f), width=850))

## 7. Modelos registrados en MLflow

Cada modelo campeón o retador se registra con un alias (`champion` o `challenger`), su firma de entrada, un
ejemplo y la etiqueta `validation_status`. Los modelos viajan con el código del proyecto, así que se cargan con
`mlflow.sklearn.load_model` sin instalar nada más.

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
cliente = MlflowClient()
registro = []
for rm in cliente.search_registered_models():
    for alias, version in rm.aliases.items():
        v = cliente.get_model_version(rm.name, version)
        corrida = cliente.get_run(v.run_id)
        registro.append({"modelo": rm.name, "alias": alias, "versión": int(version),
                         "validación": v.tags.get("validation_status"),
                         "pipeline_version": corrida.data.tags.get("pipeline_version"),
                         "git_commit": corrida.data.tags.get("git_commit")})
registro = pd.DataFrame(registro).sort_values(["alias", "modelo"], ascending=[False, True])
print(len(registro), "modelos registrados en", cfg["mlflow"]["tracking_uri"])
display(registro.reset_index(drop=True))

In [ ]:
uri = "models:/cgmt-starter-classifier@champion"
campeon = mlflow.sklearn.load_model(uri)
columnas = mlflow.models.get_model_info(uri).signature.inputs.input_names()
print("Pasos del pipeline:", [nombre for nombre, _ in campeon.steps], "| variables de entrada:", len(columnas))

# Ilustración: probabilidad de titularidad para algunos prospectos de la clase 2025. Son predicciones dentro
# de muestra (el modelo final se entrenó con todos los prospectos); la evaluación honesta es la de la sección 6.2.
prospectos = d["prospects"]
ejemplo = prospectos[prospectos.draft_year == 2025].sort_values("draft_overall_pick").head(10).copy()
ejemplo["p_titular"] = campeon.predict_proba(ejemplo[columnas])[:, 1]
display(ejemplo[["display_name", "combine_position", "draft_overall_pick", "forty", "ten_yd_split",
                 "is_starter", "p_titular"]].round(3))

In [ ]:
# Regresor del Snap Burst Index de la línea defensiva: SBI proyectado desde el Combine.
uri = "models:/cgmt-snap-burst-regressor-dl@champion"
regresor = mlflow.sklearn.load_model(uri)
columnas = mlflow.models.get_model_info(uri).signature.inputs.input_names()
dl = prospectos[prospectos.combine_position == "DL"].copy()
dl["sbi_proyectado"] = regresor.predict(dl[columnas])
display(dl.sort_values("sbi_proyectado", ascending=False)
          [["display_name", "draft_year", "combine_weight", "forty", "ten_yd_split", "sbi_proyectado"]]
          .head(10).round(2))

## 8. Siguientes pasos

- **Explorar las corridas:** en una copia local, `make ui` abre la interfaz de MLflow con todas las corridas,
  métricas por pliegue y el registro de modelos.
- **Datos nuevos:** copie los archivos nuevos en `datos/datos_ini/` (con el nombre de la especificación o como
  partes `<nombre>_<n>.csv`, comprimidos o no) y vuelva a ejecutar en modo `completa`; el contrato los valida y
  el manifiesto registra la nueva versión de datos.
- **Interpretación:** las conclusiones para scouts, oficinas generales y entrenadores están en la sección 10 de
  `CHANGELOG.md`.